
1. read bronze races table
2. keep required columns for analytics
3. standardize the column names using snake_case
4. rename columns date to _race_date
5. remove duplicate record
6. transform race_name to capitalize
7. write to silver races table

In [0]:
%run ../config/config-env

In [0]:
bronze_table = f'{catalog_name}.{bronze_schema}.races'
silver_table = f'{catalog_name}.{silver_schema}.races'

In [0]:
# read bronze table
races_df = (
        spark.read.table(bronze_table)
)
display(races_df)

In [0]:
from pyspark.sql import functions as F
races_dropped_df = races_df.select(
    F.col('season'),
    F.col('round'),
    F.col('raceName'),
    F.col('date'),
    F.col('circuitId'),
    F.col('ingestion_timestamp'),
    F.col('source_file')
)
display(races_dropped_df)

In [0]:
races_renamed_df= (
            races_dropped_df
                .withColumnsRenamed({'racename':'race_name', 'circuitId':'circuit_id'})
        )
display(races_renamed_df)

In [0]:
races_col_df = races_renamed_df.withColumnRenamed('date', 'race_date')

In [0]:
races_duplicate_removed = races_col_df.dropDuplicates(['season', 'round'])
display(races_duplicate_removed)

In [0]:
from pyspark.sql import functions as F
races_final_df = races_duplicate_removed.withColumn('race_name', F.initcap(F.col('race_name')))
display(races_final_df)

In [0]:
(
    races_final_df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(silver_table)
)